# Experiment-6: GloVe Word Embeddings and Language Modelling


## To implement the GloVe (Global Vectors for Word Representation) algorithm to generate word embeddings from the text corpus and find most similar words of “King”.

(Two separate codes 1. Without using any pretrained model 2. Using Pretrained model (Gensim) )


In [1]:
import numpy as np
import re

with open('/content/glove_corpus.txt', 'r') as file:
    corpus = file.read()

words = re.findall(r'\b\w+\b', corpus.lower())

vocab = sorted(set(words))
word_to_id = {word: i for i, word in enumerate(vocab)}

vocab_size = len(vocab)

print("Vocabulary Size:", vocab_size)

Vocabulary Size: 68


In [2]:
window_size = 3
co_matrix = np.zeros((vocab_size, vocab_size))

for i, word in enumerate(words):
    for j in range(max(0, i-window_size), min(len(words), i+window_size+1)):
        if i != j:
            w1 = word_to_id[word]
            w2 = word_to_id[words[j]]
            co_matrix[w1][w2] += 1 / abs(i-j)

print("Co-occurrence Matrix:")
print(co_matrix)

Co-occurrence Matrix:
[[2.33333333 0.5        0.         ... 1.83333333 1.         0.5       ]
 [0.5        0.         0.         ... 0.         0.         1.66666667]
 [0.         0.         0.         ... 0.         0.         0.        ]
 ...
 [1.83333333 0.         0.         ... 0.         0.         0.        ]
 [1.         0.         0.         ... 0.         0.         1.        ]
 [0.5        1.66666667 0.         ... 0.         1.         0.        ]]


In [3]:
embedding_dim = 50
learning_rate = 0.01
epochs = 100

W = np.random.randn(vocab_size, embedding_dim) * 0.01
C = np.random.randn(vocab_size, embedding_dim) * 0.01

b = np.zeros(vocab_size)
c = np.zeros(vocab_size)

x_max = 100
alpha = 0.75

for epoch in range(epochs):
    loss = 0

    for i in range(vocab_size):
        for j in range(vocab_size):
            x = co_matrix[i][j]

            if x == 0:
                continue

            weight = min((x/x_max)**alpha, 1)
            error = np.dot(W[i], C[j]) + b[i] + c[j] - np.log(x)

            loss += weight * error**2
            grad = 2 * weight * error

            old_W = W[i].copy()
            old_C = C[j].copy()

            W[i] -= learning_rate * grad * old_C
            C[j] -= learning_rate * grad * old_W

            b[i] -= learning_rate * grad
            c[j] -= learning_rate * grad

    if epoch % 20 == 0:
        print("Epoch:", epoch, "Loss:", loss)

embeddings = W + C
print("GloVe Training Completed!")

Epoch: 0 Loss: 10.227221485316099
Epoch: 20 Loss: 7.42780538581397
Epoch: 40 Loss: 6.8444249511210975
Epoch: 60 Loss: 6.564586950269162
Epoch: 80 Loss: 6.352392675090906
GloVe Training Completed!


In [4]:
def most_similar(target, top_n=5):
    target_vector = embeddings[word_to_id[target]]
    similarities = {}

    for word, idx in word_to_id.items():
        if word == target:
            continue

        vector = embeddings[idx]

        similarity = np.dot(target_vector, vector) / (
            np.linalg.norm(target_vector) * np.linalg.norm(vector) + 1e-10
        )

        similarities[word] = similarity

    return sorted(similarities.items(), key=lambda x: x[1], reverse=True)[:top_n]

print("Most Similar Words to King:")

for word, score in most_similar("king"):
    print(word, ":", round(score, 4))

Most Similar Words to King:
the : 0.6666
kingdom : 0.5046
processes : 0.2634
powerful : 0.2384
and : 0.2346


In [6]:
import gensim.downloader as api

model = api.load("glove-wiki-gigaword-100")

print("Pretrained GloVe Model Loaded Successfully!")

[==================================================] 100.0% 128.1/128.1MB downloaded
Pretrained GloVe Model Loaded Successfully!


In [7]:
similar_words = model.most_similar("king", topn=5)

print("Most Similar Words to King:")

for word, score in similar_words:
    print(word, ":", round(score, 4))

Most Similar Words to King:
prince : 0.7682
queen : 0.7508
son : 0.7021
brother : 0.6986
monarch : 0.6978


## To implement a statistical language model from scratch using the N-gram approach to predict the next word in a sequence based on the preceding words in a text corpus (shown below).

natural language processing is a branch of artificial intelligence.
natural language processing enables computers to understand human language.
machine learning is a branch of artificial intelligence.
deep learning is a part of machine learning.
language models learn patterns from text data.
a language model predicts the next word in a sentence.
natural language processing uses language models.
artificial intelligence includes machine learning and deep learning.



In [8]:
import re
from collections import Counter

corpus = """
natural language processing is a branch of artificial intelligence.
natural language processing enables computers to understand human language.
machine learning is a branch of artificial intelligence.
deep learning is a part of machine learning.
language models learn patterns from text data.
a language model predicts the next word in a sentence.
natural language processing uses language models.
artificial intelligence includes machine learning and deep learning.
"""

words = re.findall(r'\b\w+\b', corpus.lower())

print("Total Words:", len(words))
print("Words:", words)

Total Words: 65
Words: ['natural', 'language', 'processing', 'is', 'a', 'branch', 'of', 'artificial', 'intelligence', 'natural', 'language', 'processing', 'enables', 'computers', 'to', 'understand', 'human', 'language', 'machine', 'learning', 'is', 'a', 'branch', 'of', 'artificial', 'intelligence', 'deep', 'learning', 'is', 'a', 'part', 'of', 'machine', 'learning', 'language', 'models', 'learn', 'patterns', 'from', 'text', 'data', 'a', 'language', 'model', 'predicts', 'the', 'next', 'word', 'in', 'a', 'sentence', 'natural', 'language', 'processing', 'uses', 'language', 'models', 'artificial', 'intelligence', 'includes', 'machine', 'learning', 'and', 'deep', 'learning']


In [9]:
unigrams = Counter(words)

bigrams = Counter(zip(words, words[1:]))

trigrams = Counter(zip(words, words[1:], words[2:]))

print("Unigram Frequencies:")
print(unigrams)

print("\nBigram Frequencies:")
print(bigrams)

print("\nTrigram Frequencies:")
print(trigrams)

Unigram Frequencies:
Counter({'language': 7, 'a': 5, 'learning': 5, 'natural': 3, 'processing': 3, 'is': 3, 'of': 3, 'artificial': 3, 'intelligence': 3, 'machine': 3, 'branch': 2, 'deep': 2, 'models': 2, 'enables': 1, 'computers': 1, 'to': 1, 'understand': 1, 'human': 1, 'part': 1, 'learn': 1, 'patterns': 1, 'from': 1, 'text': 1, 'data': 1, 'model': 1, 'predicts': 1, 'the': 1, 'next': 1, 'word': 1, 'in': 1, 'sentence': 1, 'uses': 1, 'includes': 1, 'and': 1})

Bigram Frequencies:
Counter({('natural', 'language'): 3, ('language', 'processing'): 3, ('is', 'a'): 3, ('artificial', 'intelligence'): 3, ('machine', 'learning'): 3, ('a', 'branch'): 2, ('branch', 'of'): 2, ('of', 'artificial'): 2, ('learning', 'is'): 2, ('deep', 'learning'): 2, ('language', 'models'): 2, ('processing', 'is'): 1, ('intelligence', 'natural'): 1, ('processing', 'enables'): 1, ('enables', 'computers'): 1, ('computers', 'to'): 1, ('to', 'understand'): 1, ('understand', 'human'): 1, ('human', 'language'): 1, ('languag

In [10]:
def predict_next_word(context):
    context = tuple(context.lower().split())

    candidates = {}

    if len(context) >= 2:
        last_two = context[-2:]

        for (w1, w2, w3), count in trigrams.items():
            if (w1, w2) == last_two:
                candidates[w3] = count / bigrams[(w1, w2)]

    if not candidates and len(context) >= 1:
        last_word = context[-1]

        for (w1, w2), count in bigrams.items():
            if w1 == last_word:
                candidates[w2] = count / unigrams[w1]

    if not candidates:
        return "No prediction available"

    return max(candidates, key=candidates.get)


print("Next Word Predictions:\n")

test_sentences = [
    "natural language",
    "machine learning",
    "artificial intelligence",
    "deep learning"
]

for sentence in test_sentences:
    print("Input:", sentence)
    print("Predicted Next Word:", predict_next_word(sentence))
    print()

Next Word Predictions:

Input: natural language
Predicted Next Word: processing

Input: machine learning
Predicted Next Word: is

Input: artificial intelligence
Predicted Next Word: natural

Input: deep learning
Predicted Next Word: is

